# Refusal vs Scratch Sampling Analysis

This notebook analyzes whether sampling from scratch is equivalent to sampling after a refusal across different tactics and batches.

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from typing import List, Tuple, Dict
import glob

plt.style.use('default')
sns.set_palette('husl')

In [2]:
def load_batch_data(batch_name: str, tactic: str) -> pd.DataFrame:
    """
    Load data for a specific batch and tactic combination.
    """
    pattern = f"clean_results/final_runs/{batch_name}/strongreject_evaluator_*{tactic}*.jsonl"
    files = glob.glob(pattern)
    
    if not files:
        print(f"No files found for pattern: {pattern}")
        return pd.DataFrame()
    
    data = []
    for file in files:
        with open(file, 'r') as f:
            for line in f:
                try:
                    entry = json.loads(line.strip())
                    data.append(entry)
                except json.JSONDecodeError:
                    continue
    
    return pd.DataFrame(data)

In [3]:
def extract_first_and_subsequent_responses(df: pd.DataFrame) -> Tuple[List[float], List[float]]:
    """
    Extract first responses (scratch sampling) and subsequent responses (after refusal).
    """
    first_responses = []
    subsequent_responses = []
    
    for _, row in df.iterrows():
        if 'num_turns' in row and row['num_turns'] != 1:
            continue
            
        scores = row['score']
        if not isinstance(scores, list) or len(scores) == 0:
            continue
            
        # First response (scratch sampling)
        if isinstance(scores[0], (int, float)):
            first_responses.append(float(scores[0]))
        
        # Subsequent responses (after refusal)
        for score in scores[1:]:
            if isinstance(score, (int, float)):
                subsequent_responses.append(float(score))
    
    return first_responses, subsequent_responses

In [4]:
def plot_comparison_histograms(first_scores: List[float], subsequent_scores: List[float], 
                             title: str, ax=None) -> Dict:
    """
    Plot comparison histograms and return statistics.
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    bins = np.linspace(0, 1, 21)
    
    ax.hist(first_scores, bins=bins, alpha=0.7, label=f'First (Scratch) n={len(first_scores)}', 
            color='skyblue', edgecolor='black')
    ax.hist(subsequent_scores, bins=bins, alpha=0.7, label=f'Subsequent (After Refusal) n={len(subsequent_scores)}', 
            color='lightcoral', edgecolor='black')
    
    ax.set_xlabel('StrongREJECT Score')
    ax.set_ylabel('Frequency')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Statistical test
    if len(first_scores) > 0 and len(subsequent_scores) > 0:
        statistic, p_value = stats.mannwhitneyu(first_scores, subsequent_scores, alternative='two-sided')
        
        # Summary statistics
        stats_dict = {
            'first_mean': np.mean(first_scores),
            'first_median': np.median(first_scores),
            'subsequent_mean': np.mean(subsequent_scores),
            'subsequent_median': np.median(subsequent_scores),
            'p_value': p_value,
            'n_first': len(first_scores),
            'n_subsequent': len(subsequent_scores)
        }
        
        # Add statistics text to plot
        stats_text = f"First: μ={stats_dict['first_mean']:.3f}, med={stats_dict['first_median']:.3f}\n"
        stats_text += f"Subsequent: μ={stats_dict['subsequent_mean']:.3f}, med={stats_dict['subsequent_median']:.3f}\n"
        stats_text += f"Mann-Whitney U p-value: {p_value:.4f}"
        
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        return stats_dict
    
    return {}

In [5]:
def analyze_single_case(batch_name: str, tactic: str) -> Dict:
    """
    Analyze a single batch-tactic combination.
    """
    print(f"\nAnalyzing {batch_name} - {tactic}")
    print("=" * 40)
    
    df = load_batch_data(batch_name, tactic)
    if df.empty:
        print("No data found")
        return {}
    
    first_scores, subsequent_scores = extract_first_and_subsequent_responses(df)
    
    print(f"Found {len(first_scores)} first responses and {len(subsequent_scores)} subsequent responses")
    
    if len(first_scores) == 0 or len(subsequent_scores) == 0:
        print("Insufficient data for comparison")
        return {}
    
    title = f"{batch_name} - {tactic}: First vs Subsequent Responses"
    stats_dict = plot_comparison_histograms(first_scores, subsequent_scores, title)
    plt.show()
    
    return stats_dict

## Individual Case Analysis

Let's analyze each of the 4 combinations separately:

In [6]:
# Case 1: batch6A + direct_request
stats_6A_direct = analyze_single_case("batch6A", "direct_request")


Analyzing batch6A - direct_request
No files found for pattern: clean_results/final_runs/batch6A/strongreject_evaluator_*direct_request*.jsonl
No data found


In [7]:
# Case 2: batch6A + command
stats_6A_command = analyze_single_case("batch6A", "command")


Analyzing batch6A - command
No files found for pattern: clean_results/final_runs/batch6A/strongreject_evaluator_*command*.jsonl
No data found


In [8]:
# Case 3: batch6B + direct_request
stats_6B_direct = analyze_single_case("batch6B", "direct_request")


Analyzing batch6B - direct_request
No files found for pattern: clean_results/final_runs/batch6B/strongreject_evaluator_*direct_request*.jsonl
No data found


In [9]:
# Case 4: batch6B + command
stats_6B_command = analyze_single_case("batch6B", "command")


Analyzing batch6B - command
No files found for pattern: clean_results/final_runs/batch6B/strongreject_evaluator_*command*.jsonl
No data found


## Combined Analysis

Now let's combine all cases into a single analysis:

In [10]:
def analyze_combined_cases():
    """
    Combine all 4 cases and analyze together.
    """
    print("\nCombined Analysis (All Cases)")
    print("=" * 40)
    
    all_first_scores = []
    all_subsequent_scores = []
    
    cases = [
        ("batch6A", "direct_request"),
        ("batch6A", "command"),
        ("batch6B", "direct_request"),
        ("batch6B", "command")
    ]
    
    for batch_name, tactic in cases:
        df = load_batch_data(batch_name, tactic)
        if not df.empty:
            first_scores, subsequent_scores = extract_first_and_subsequent_responses(df)
            all_first_scores.extend(first_scores)
            all_subsequent_scores.extend(subsequent_scores)
    
    print(f"Combined: {len(all_first_scores)} first responses and {len(all_subsequent_scores)} subsequent responses")
    
    if len(all_first_scores) > 0 and len(all_subsequent_scores) > 0:
        title = "Combined Analysis: First vs Subsequent Responses (All Cases)"
        combined_stats = plot_comparison_histograms(all_first_scores, all_subsequent_scores, title)
        plt.show()
        return combined_stats
    
    return {}

combined_stats = analyze_combined_cases()


Combined Analysis (All Cases)
No files found for pattern: clean_results/final_runs/batch6A/strongreject_evaluator_*direct_request*.jsonl
No files found for pattern: clean_results/final_runs/batch6A/strongreject_evaluator_*command*.jsonl
No files found for pattern: clean_results/final_runs/batch6B/strongreject_evaluator_*direct_request*.jsonl
No files found for pattern: clean_results/final_runs/batch6B/strongreject_evaluator_*command*.jsonl
Combined: 0 first responses and 0 subsequent responses


## Summary and Conclusions

In [11]:
def generate_summary_table():
    """
    Generate a summary table of all results.
    """
    all_stats = {
        "batch6A_direct": stats_6A_direct,
        "batch6A_command": stats_6A_command,
        "batch6B_direct": stats_6B_direct,
        "batch6B_command": stats_6B_command,
        "combined": combined_stats
    }
    
    print("\nSummary Table")
    print("=" * 80)
    print(f"{'Case':<15} {'First Mean':<12} {'Subseq Mean':<12} {'First Med':<12} {'Subseq Med':<12} {'p-value':<10} {'Significant':<12}")
    print("-" * 80)
    
    for case_name, stats in all_stats.items():
        if stats:
            significant = "✓" if stats['p_value'] < 0.05 else "✗"
            print(f"{case_name:<15} {stats['first_mean']:<12.3f} {stats['subsequent_mean']:<12.3f} "
                  f"{stats['first_median']:<12.3f} {stats['subsequent_median']:<12.3f} "
                  f"{stats['p_value']:<10.4f} {significant:<12}")
    
    print("\nConclusions:")
    print("- ✓ indicates statistically significant difference (p < 0.05)")
    print("- ✗ indicates no significant difference (sampling equivalence)")

generate_summary_table()


Summary Table
Case            First Mean   Subseq Mean  First Med    Subseq Med   p-value    Significant 
--------------------------------------------------------------------------------

Conclusions:
- ✓ indicates statistically significant difference (p < 0.05)
- ✗ indicates no significant difference (sampling equivalence)
